# FPGA Security Evaluation — ZedBoard Rev E

This notebook documents the **qualitative security evaluation** of the ZedBoard Revision E used for local and cloud-assisted FPGA inference in the dissertation, *Machine Learning on the Edge: Performance and Security Evaluation of Artificial Neural Network Implementations in Embedded Systems*.

The purpose is to preserve the evaluation sequence, configuration evidence, source-code observations, applicable vendor notices, and limitations. This document supports Chapter IV, Section 4.7.2.1 (*FPGA Deployment*). The detailed Ethernet, TLS, and Google Cloud Run review belongs to the separate cloud-assisted security evaluation.

This is a record of configuration and documentation review, **not** a record of penetration testing.

## 1. Experimental Platform

| Component | Actual configuration |
| --- | --- |
| FPGA board | ZedBoard Rev E |
| SoC | Xilinx Zynq-7020 (`XC7Z020CLG484-1`) |
| Development software | Vivado 2018.1 and Xilinx SDK 2018.1 |
| Hardware programming | JTAG through Vivado |
| Local classification | zyNet NN implemented in programmable logic (PL) |
| Cloud-assisted client | ARM processing system (PS) application launched separately through SDK |
| Camera | OV7670 |
| Development workstation | HP ZBook Fury 17 G7 running Windows 11 Pro |
| Review type | Screenshot inspection, source-configuration review, access-condition documentation, and advisory applicability screening |

The implementation files reviewed for model loading and acquisition logic are maintained in the **MNIST FPGA Live Classification Code** branch. Official manuals, specifications, and AMD security references are catalogued in the **Manuals and Specifications** branch. The screenshots examined below form the direct evidence for this security evaluation.

## 2. Evaluation Sequence

The review followed these steps:

```text
1. Identify hardware and software versions.
2. Review the JTAG programming and bitstream generation log.
3. Inspect the Vivado bitstream settings for encryption/authentication evidence.
4. Document physical, UART/JTAG, workstation, and remote access conditions.
5. Examine HDL compile-time controls for weights and biases.
6. Screen AMD/Xilinx and NVD notices against actual hardware and boot procedures.
7. Review Cortex-A9 architectural vulnerability disclosures.
8. Review Vivado 2018.1 IP release-note entries.
9. Examine the OV7670 acquisition interface and relevant literature.
10. Record findings without claiming unperformed attacks or measurements.
```

Each notice was assessed by *affected component, affected version, required attack conditions, and observed experimental configuration*.

## 3. JTAG Programming and Bitstream Generation

Both FPGA configurations were programmed using Vivado 2018.1 and JTAG. For cloud-assisted inference, the ARM communication application was launched separately through Xilinx SDK 2018.1.

The reviewed Vivado log recorded the standard command:

```tcl
write_bitstream -force design_1_wrapper.bit
```

The successful bitstream generation confirmed that a programmable design was produced; it did **not**, by itself, establish that secure boot, encryption, or authentication was enabled.

**Screenshot evidence: `write_bitstream_log.png`**

![Vivado bitstream generation log](write_bitstream_log.png)

The relevant hardware security capabilities are documented in [Zynq-7000 SoC Technical Reference Manual (UG585)](https://docs.amd.com/r/en-US/ug585-zynq-7000-SoC-TRM/Introduction?tocId=Hf6C7Oo5ABvv2hkWRoiihQ).

## 4. Vivado Bitstream Security Settings

The bitstream settings were reviewed for identifiable encryption and authentication properties.

**Screenshot evidence: `bitstream_settings.png`**

![Vivado bitstream settings](bitstream_settings.png)

No settings establishing that bitstream encryption or authentication was enabled were identified in the reviewed properties. This finding is limited to the captured configuration evidence and should **not** be represented as a cryptographic inspection of the generated `.bit` file.

Direct JTAG access remained a potential reconfiguration pathway if an attacker could control the physical interface or development environment. This was a threat inference rather than a demonstrated exploit.

## 5. Physical and Debugging Access

The documented operating conditions were:

- The ZedBoard was operated in a **private office**, limiting casual physical access.
- **JTAG and UART** were connected during most development and testing.
- JTAG and UART were disconnected for power measurements and demonstrations of self-contained local or cloud-assisted classification.
- The Windows workstation required a PIN at sign-in; Vivado and Xilinx SDK were accessible to users of the active authenticated session.
- A **different** Meta Quest 3S from the headset used in experiments was configured with its built-in Remote Desktop application to control an active workstation session.

The remote desktop arrangement provided another possible path to the workstation and its connected development tools. The evaluation did not establish unauthorized access or successful reprogramming.

## 6. Model-Parameter Initialization — Source Review

The FPGA model generation and RTL files reviewed under the **MNIST FPGA Live Classification Code** branch included the zyNet generation script, `Weight_Memory.v`, `neuron.v`, and `include.v`.

The implementation used pretrained weights and biases supplied through initialization files. The reviewed `include.v` defined:

```verilog
`define pretrained
```

The weight memory selected an initialization path using `$readmemb`, while the alternative normal writable branch was excluded when `pretrained` was active. Bias initialization in `neuron.v` used the corresponding conditional compilation approach.

**Interpretation:** The reviewed compiled source path excluded the normal external weight/bias update mechanisms during inference. Parameter integrity still depended on the source files, generated data, and programmed hardware configuration. No attempts were made to extract or manipulate model parameters.

## 7. Advisory Applicability Criteria

Each source was screened using the following questions:

1. Does the issue concern **Zynq-7000 / Cortex-A9**, or another device family?
2. Does it concern **Vivado / Xilinx SDK 2018.1**, or a software version not established as relevant?
3. What execution, boot, programming, or authentication path is required?
4. Was that path documented during the experiments?
5. Is the conclusion a confirmed architectural susceptibility, a non-matching attack path, or an unresolved applicability issue?

**Interpretation terms:**

- *Architectural susceptibility* — the affected processor architecture was present; exploitability was not tested.
- *Attack prerequisites not established* — the documented procedure does not demonstrate use of the vulnerable mechanism.
- *Different device family* — the notice does not target Zynq-7020 hardware.
- *Version applicability unverified* — the source did not establish that 2018.1 was affected.
- *Release documentation* — release-note entries are not automatically vulnerability disclosures.

## 8. Vendor Security Reference Index

The **Manuals and Specifications** branch contains the consolidated official online reference list. The security sources examined for this FPGA implementation were:

- [AMD UG585 — Zynq-7000 SoC Technical Reference Manual](https://docs.amd.com/r/en-US/ug585-zynq-7000-SoC-TRM/Introduction?tocId=Hf6C7Oo5ABvv2hkWRoiihQ)
- [AMD 70394 — Meltdown and Spectre on Zynq devices](https://adaptivesupport.amd.com/s/article/70394?language=en_US)
- [AMD 71292 — Zynq-7000 FSBL security advisory](https://adaptivesupport.amd.com/s/article/71292?language=en_US)
- [AMD 76974 — FSBL authentication attack](https://adaptivesupport.amd.com/s/article/76974?language=en_US)
- [AMD 73541 — Defeating bitstream encryption](https://docs.amd.com/r/en-US/73541/Solution)
- [AMD 70699 — Vivado 2018.1 IP change log](https://adaptivesupport.amd.com/s/article/70699?language=en_US)
- [OpenCVE — Zynq-7000 records](https://app.opencve.io/cve/?product=zynq-7000&vendor=xilinx)
- [NVD — CVE-2021-44850](https://nvd.nist.gov/vuln/detail/CVE-2021-44850)
- [NVD — CVE-2021-27208](https://nvd.nist.gov/vuln/detail/CVE-2021-27208)
- [NVD — CVE-2022-23822](https://nvd.nist.gov/vuln/detail/CVE-2022-23822)

These sources were examined for **applicability**, not assumed to describe successful attacks on this platform.

## 9. Boot and Bitstream Advisories

| Source | Reported issue | Evaluation observation |
| --- | --- | --- |
| [CVE-2021-44850](https://nvd.nist.gov/vuln/detail/CVE-2021-44850) | Manipulated SD boot image / boot ROM | The documented execution used JTAG, not SD-card boot. |
| [CVE-2021-27208](https://nvd.nist.gov/vuln/detail/CVE-2021-27208) | NAND boot-related issue | NAND boot was not part of the documented workflow. |
| [AMD 71292](https://adaptivesupport.amd.com/s/article/71292?language=en_US) | FSBL partition-header security handling in 2018.2 and earlier | No authenticated FSBL boot workflow was established. |
| [AMD 76974](https://adaptivesupport.amd.com/s/article/76974?language=en_US) | FSBL authentication attack | The affected authentication procedure was not documented. |
| [AMD 73541](https://docs.amd.com/r/en-US/73541/Solution) | Attacks against affected encrypted 7-Series bitstreams | Encryption was not identified in reviewed settings. |

No `fsbl.elf` or `BOOT.BIN` files were found in the project examined during this review. Their absence does **not conclusively exclude** all possible bootloader or initialization activity. The conclusion is limited to the documented and inspected workflow.

## 10. Cortex-A9 Spectre Susceptibility

AMD [Answer Record 70394](https://adaptivesupport.amd.com/s/article/70394?language=en_US) reports that ARM Cortex-A9 can be susceptible to:

- **CVE-2017-5753:** Spectre Variant 1 — bounds check bypass.
- **CVE-2017-5715:** Spectre Variant 2 — branch target injection.

The Zynq-7020 contains Cortex-A9 processors. Thus, **processor-level architectural susceptibility** was relevant to the experimental hardware.

For local FPGA classification, the neural network executed in PL without the ARM PS performing classification. The PS executed the network client in the cloud-assisted configuration. The advisory is therefore especially pertinent to the ARM software execution environment, but the review did not test exploitability, memory leakage, or execution of speculative-execution attacks.

## 11. Vivado and Xilinx SDK Documentation Review

The exact tool versions were **Vivado 2018.1** and **Xilinx SDK 2018.1**.

Relevant vendor references included:

- [AMD 70699 — Vivado 2018.1 IP Release Notes](https://adaptivesupport.amd.com/s/article/70699?language=en_US)
- [Vivado 2018.1 Release Notes (UG973)](https://docs.amd.com/v/u/2018.1-English/ug973-vivado-release-notes-install-license)
- [Xilinx SDK/XSCT 2018.1 (UG1208)](https://docs.amd.com/v/u/2018.1-English/ug1208-xsct-reference-guide)

In the reviewed IP Status report, ILA **6.2** corresponded to a release-note entry updating XDC warning handling. The other reviewed IP entries were listed as having no change.

**Conclusion:** The compared IP release-note entries did not identify a security-related change to those components. This does **not** prove the installed IP was vulnerability-free.

[NVD CVE-2025-52541](https://nvd.nist.gov/vuln/detail/CVE-2025-52541) was also reviewed but its applicability to **Vivado 2018.1** was not established; it was not classified as a confirmed finding.

## 12. Device-Family Screening

[AMD 66071 — Zynq UltraScale+ MPSoC Design Advisory Master Record](https://adaptivesupport.amd.com/s/article/66071?language=en_US) concerns the **UltraScale+** product family.

The evaluated ZedBoard used the **Zynq-7000 Zynq-7020**, not an UltraScale+ MPSoC. This notice was excluded as evidence of a vulnerability affecting the experimental SoC.

## 13. OV7670 Camera Security

The OV7670 supplied image data through a direct hardware interface to the PL acquisition/preprocessing logic. The literature reference located for comparison was:

- [Liu et al. (2024), *Real-Time Secure Video Streaming System Based on FPGA and CUDA Technology*](https://doi.org/10.1145/3711618.3711642)

Liu et al. used an **OV7670**, a **Nexys A7-100T FPGA**, BRAM, and FPGA-side encryption before Ethernet transmission. Their paper demonstrates a protective camera-data pipeline, **not** a documented vulnerability of the OV7670.

The reviewed literature did not identify an OV7670-specific vulnerability. Potential concerns were unauthorized manipulation of the physical camera connection, image inputs, or acquisition logic. No camera-specific attack was performed.

## 14. Consolidated Findings

| Area | Evidence-based observation | Security implication or limitation |
| --- | --- | --- |
| Bitstream / JTAG | Direct JTAG programming; security properties not identified in screenshots | Potential reconfiguration with programming access; protection not verified enabled |
| Physical/debug access | Private office; JTAG/UART commonly connected | Limited casual access, but interfaces exposed during development |
| Remote desktop | Separate Quest headset permitted control of active workstation session | Additional possible access route to tools; unauthorized use not demonstrated |
| Model parameters | `pretrained` RTL initialization excluded alternative write branches | Normal parameter-update path constrained; source and programming integrity still important |
| SD/NAND/FSBL notices | Vulnerable boot/authentication paths not established in reviewed workflow | No matching exploit demonstrated; absence of files alone not definitive |
| Cortex-A9 | Spectre Variant 1/2 architectural susceptibility | Relevant particularly to PS software; no exploit test |
| Vivado IP | ILA 6.2 change related to XDC warnings; others unchanged in reviewed notes | No security-related change identified in examined entries |
| OV7670 | Camera-specific CVE not identified in source reviewed | Potential physical/input threats, not a documented sensor exploit |

## 15. Evidence Index

| Evidence or reference | Role in evaluation | Where documented |
| --- | --- | --- |
| `write_bitstream_log.png` | Build/programming workflow | FPGA Security branch |
| `bitstream_settings.png` | Bitstream property review | FPGA Security branch |
| zyNet generation script | Parameter file and pretrained configuration | MNIST FPGA Live Classification Code branch |
| `Weight_Memory.v` | Conditional weight initialization/write paths | MNIST FPGA Live Classification Code branch |
| `include.v` | Active `pretrained` compile-time definition | MNIST FPGA Live Classification Code branch |
| `neuron.v` | Bias initialization and alternative update path | MNIST FPGA Live Classification Code branch |
| AMD manuals and advisories | Device capabilities and vulnerability applicability | Manuals and Specifications branch |
| Liu et al. (2024) | Related OV7670/FPGA encryption design | Official DOI listed above |

The screenshots and code review support different conclusions. No source-code finding is inferred solely from a screenshot.

## 16. Reproduction Sequence

A repeat evaluation of the same type would:

```text
[ ] Identify exact board, SoC, Vivado, and SDK versions.
[ ] Inspect JTAG programming steps and the bitstream generation log.
[ ] Review actual Vivado bitstream security settings.
[ ] Record physical access, JTAG/UART attachment, and workstation access.
[ ] Inspect source defines, weight initialization, and bias initialization paths.
[ ] Screen advisories by affected device, version, and prerequisite attack path.
[ ] Inspect actual processor architecture and vendor disclosures.
[ ] Compare documented IP component versions to 2018.1 release notes.
[ ] Review camera interface and supporting literature.
[ ] Preserve findings and any uncertainty about the execution path.
```

This is a **documentation and configuration review**, not instructions for recreating an unauthorized intrusion or a measured attack.

## 17. Reporting Limits and Dissertation Use

The investigation did not perform penetration tests, model extraction, fault injection, malicious bitstream loading, camera attacks, Spectre demonstrations, or independent cryptographic verification.

No runtime security-overhead measurements were taken for features that were not identified as active. A vulnerability record for a device family was not treated as proof that its specific exploitation prerequisites existed.

The notebook preserves the supporting evaluation evidence for **Chapter IV, Section 4.7.2.1**. Cross-platform risk interpretation and performance/security trade-offs are addressed separately in the dissertation.